In [1]:
import os
import time
import json
import torch
import librosa
import numpy as np
import itertools
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.tensorboard import SummaryWriter
from transformers import Wav2Vec2Model

ROOT_DIR = Path(os.getcwd()).parent
MODELS_DIR = ROOT_DIR / "models"
LOGS_DIR = ROOT_DIR / "logs"

for d in [MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/home/daniellu/Documents/GitHub/mime/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
class AudioToBlendshapeModel(torch.nn.Module):
    def __init__(self, num_blendshapes=52, hidden_dim=512, use_pretrained=False):
        super().__init__()
        self.use_pretrained = use_pretrained
        
        if self.use_pretrained:
            # Mode A: Pre-trained Phonetic Encoder (Wav2Vec 2.0)
            self.audio_backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
            self.audio_proj = torch.nn.Linear(768, hidden_dim)
        else:
            # Mode B: Custom Phonetic-CNN (No pre-training)
            self.audio_backbone = torch.nn.Sequential(
                torch.nn.Conv1d(1, 64, kernel_size=10, stride=5),
                torch.nn.BatchNorm1d(64),
                torch.nn.ReLU(),
                torch.nn.Conv1d(64, 128, kernel_size=3, stride=2),
                torch.nn.ReLU(),
                torch.nn.Conv1d(128, 256, kernel_size=3, stride=2),
                torch.nn.ReLU(),
                torch.nn.Conv1d(256, hidden_dim, kernel_size=3, stride=2),
                torch.nn.ReLU()
            )

        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, batch_first=True
        )
        self.transformer = torch.nn.TransformerEncoder(encoder_layer, num_layers=4)
        
        self.regressor = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 256),
            torch.nn.LeakyReLU(0.2),
            torch.nn.Linear(256, num_blendshapes),
            torch.nn.Sigmoid() 
        )

    def forward(self, x):
        if self.use_pretrained:
            # Wav2Vec expects (Batch, Samples)
            features = self.audio_backbone(x).last_hidden_state
            x = self.audio_proj(features)
        else:
            # CNN expects (Batch, Channels, Samples)
            if x.dim() == 2: x = x.unsqueeze(1)
            x = self.audio_backbone(x).transpose(1, 2)
        
        x = self.transformer(x)
        return self.regressor(x)

In [3]:
class BEATDataset(Dataset):
    def __init__(self, beat_root, target_sr=16000, max_samples=None):
        self.beat_root = Path(beat_root)
        self.target_sr = target_sr
        self.items = []

        wav_candidates = sorted(self.beat_root.rglob("*.wav"))
        for wav_path in wav_candidates:
            json_path = wav_path.with_suffix(".json")
            if json_path.exists():
                self.items.append((wav_path, json_path))

        if max_samples is not None:
            self.items = self.items[:max_samples]

        if len(self.items) == 0:
            raise FileNotFoundError(
                f"No BEAT wav+json pairs found under {self.beat_root}. "
                "Ensure files like <sample>.wav and <sample>.json exist side-by-side."
            )

    def __len__(self):
        return len(self.items)

    def _load_target(self, target_path):
        with open(target_path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        frames = []
        if isinstance(obj, dict):
            if isinstance(obj.get("frames"), list):
                frames = obj["frames"]
            elif isinstance(obj.get("weights"), list):
                frames = [{"weights": obj["weights"]}]
        elif isinstance(obj, list):
            frames = obj

        weights = []
        for fr in frames:
            if isinstance(fr, dict) and isinstance(fr.get("weights"), list):
                weights.append(fr["weights"])
            elif isinstance(fr, list):
                weights.append(fr)

        if len(weights) == 0:
            raise ValueError(f"No frame weights in {target_path}")

        tgt = torch.tensor(weights, dtype=torch.float32)

        if tgt.dim() == 1:
            tgt = tgt.unsqueeze(0)
        elif tgt.dim() > 2:
            tgt = tgt.reshape(tgt.shape[0], -1)

        feat = tgt.shape[1]
        if feat < 52:
            pad = torch.zeros(tgt.shape[0], 52 - feat, dtype=tgt.dtype)
            tgt = torch.cat([tgt, pad], dim=1)
        elif feat > 52:
            tgt = tgt[:, :52]

        return tgt

    def __getitem__(self, idx):
        audio_path, target_path = self.items[idx]
        audio_np, sr = librosa.load(audio_path, sr=None, mono=True)
        if sr != self.target_sr:
            audio_np = librosa.resample(audio_np, orig_sr=sr, target_sr=self.target_sr)
        audio = torch.tensor(audio_np, dtype=torch.float32)

        target = self._load_target(target_path)
        return audio, target


def collate_fn(batch):
    audio_seqs, target_seqs = zip(*batch)
    audio_padded = pad_sequence(audio_seqs, batch_first=True, padding_value=0.0)
    target_padded = pad_sequence(target_seqs, batch_first=True, padding_value=0.0)
    return audio_padded, target_padded

In [4]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for audio, target in loader:
        if audio.numel() == 0 or target.numel() == 0:
            continue

        audio = audio.to(device)
        target = target.to(device)
        optimizer.zero_grad()

        output = model(audio)  # [Batch, Seq_Len_A, 52]

        if output.shape[1] != target.shape[1]:
            output = output.transpose(1, 2)
            output = F.interpolate(output, size=target.shape[1], mode="linear", align_corners=False)
            output = output.transpose(1, 2)

        mse = criterion(output, target)
        vel_loss = criterion(output[:, 1:] - output[:, :-1], target[:, 1:] - target[:, :-1])
        loss = mse + 0.5 * vel_loss

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(1, num_batches)


def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    total_mae = 0.0
    num_batches = 0

    with torch.no_grad():
        for audio, target in loader:
            if audio.numel() == 0 or target.numel() == 0:
                continue

            audio = audio.to(device)
            target = target.to(device)
            output = model(audio)

            if output.shape[1] != target.shape[1]:
                output = output.transpose(1, 2)
                output = F.interpolate(output, size=target.shape[1], mode="linear", align_corners=False)
                output = output.transpose(1, 2)

            loss = criterion(output, target)
            mae = F.l1_loss(output, target)

            if not (torch.isfinite(loss) and torch.isfinite(mae)):
                continue

            val_loss += loss.item()
            total_mae += mae.item()
            num_batches += 1

    return val_loss / max(1, num_batches), total_mae / max(1, num_batches)


def run_trial(config, train_loader, val_loader):
    run_id = f"TRIAL_{int(time.time())}_pre{config['use_pretrained']}_h{config['hidden_dim']}"
    writer = SummaryWriter(log_dir=LOGS_DIR / run_id)

    model = AudioToBlendshapeModel(
        hidden_dim=config["hidden_dim"],
        use_pretrained=config["use_pretrained"],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = torch.nn.MSELoss()
    best_val_loss = float("inf")

    for epoch in range(config["epochs"]):
        t_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss, v_mae = validate(model, val_loader, criterion, device)

        print(
            f"Epoch {epoch+1}/{config['epochs']} -> "
            f"Train Loss: {t_loss:.6f}, Val Loss: {v_loss:.6f}, Val MAE: {v_mae:.6f}"
        )

        writer.add_scalars("Loss/Combined", {"train": t_loss, "val": v_loss}, epoch)
        writer.add_scalar("Accuracy/Val_MAE", v_mae, epoch)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": config,
                    "val_loss": best_val_loss,
                },
                MODELS_DIR / f"{run_id}_best.pt",
            )

    writer.add_hparams(config, {"hparam/best_val_loss": best_val_loss})
    writer.close()
    return best_val_loss

In [5]:
# Local BEAT dataset root
BEAT_ROOT = "../data/beat_english_v0.2.1/"

# Set to an integer like 240 for quick experiments
MAX_SAMPLES = None

full_dataset = BEATDataset(BEAT_ROOT, target_sr=16000, max_samples=MAX_SAMPLES)
print(f"Loaded BEAT wav+json pairs: {len(full_dataset)}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

# Quick batch sanity check
sanity_loader = DataLoader(train_set, batch_size=4, collate_fn=collate_fn)
audio_b, target_b = next(iter(sanity_loader))
print(f"Batch Audio: {audio_b.shape}")
print(f"Batch Targets: {target_b.shape}")

search_space = {
    "lr": [1e-4, 5e-5],
    "hidden_dim": [256, 512],
    "use_pretrained": [True, False],
    "batch_size": [4],
    "epochs": [5, 20],
}

keys, values = zip(*search_space.items())
trials = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total Trials to Run: {len(trials)}")

Loaded BEAT wav+json pairs: 1945
Batch Audio: torch.Size([4, 10032000])
Batch Targets: torch.Size([4, 37619, 52])
Total Trials to Run: 16


In [ ]:
for i, cfg in enumerate(trials):
    print(f"\n--- [Trial {i+1}/{len(trials)}] Config: {cfg} ---")

    t_loader = DataLoader(train_set, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate_fn)
    v_loader = DataLoader(val_set, batch_size=cfg["batch_size"], shuffle=False, collate_fn=collate_fn)

    try:
        res = run_trial(cfg, t_loader, v_loader)
        print(f"Success. Best Val Loss: {res:.6f}")
    except Exception as e:
        print(f"Trial crashed: {e}")


--- [Trial 1/16] Config: {'lr': 0.0001, 'hidden_dim': 256, 'use_pretrained': True, 'batch_size': 4, 'epochs': 5} ---


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 2656.21it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/5 -> Train Loss: 0.167808, Val Loss: 0.083397, Val MAE: 0.250149
Epoch 2/5 -> Train Loss: 0.167154, Val Loss: 0.083379, Val MAE: 0.250090
Epoch 3/5 -> Train Loss: 0.167068, Val Loss: 0.083324, Val MAE: 0.249925
Epoch 4/5 -> Train Loss: 0.166935, Val Loss: 0.083939, Val MAE: 0.251251
Epoch 5/5 -> Train Loss: 0.166826, Val Loss: 0.082929, Val MAE: 0.249141
Success. Best Val Loss: 0.082929

--- [Trial 2/16] Config: {'lr': 0.0001, 'hidden_dim': 256, 'use_pretrained': True, 'batch_size': 4, 'epochs': 20} ---


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 1302.05it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 2/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 3/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 4/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 5/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 6/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 7/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 8/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 9/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 10/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 11/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 12/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 13/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 14/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 15/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 16/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 17/20 -> Train Loss: nan, Val Loss: nan, Va

...

Epoch 1/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 2/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 3/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 4/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 5/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 6/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 7/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 8/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 9/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 10/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 11/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 12/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 13/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 14/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 15/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 16/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 17/20 -> Train Loss: nan, Val Loss: nan, Va

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3467.38it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/5 -> Train Loss: 0.167734, Val Loss: 0.083039, Val MAE: 0.249236
Epoch 2/5 -> Train Loss: 0.167158, Val Loss: 0.083915, Val MAE: 0.250921
Epoch 3/5 -> Train Loss: 0.167338, Val Loss: 0.083127, Val MAE: 0.249703
Epoch 4/5 -> Train Loss: 0.167413, Val Loss: 0.083507, Val MAE: 0.250356
Epoch 5/5 -> Train Loss: 0.166626, Val Loss: 0.083231, Val MAE: 0.249651
Success. Best Val Loss: 0.083039

--- [Trial 14/16] Config: {'lr': 5e-05, 'hidden_dim': 512, 'use_pretrained': True, 'batch_size': 4, 'epochs': 20} ---


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3003.66it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 2/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 3/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 4/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 5/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 6/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 7/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 8/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 9/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 10/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 11/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 12/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 13/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 14/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 15/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 16/20 -> Train Loss: nan, Val Loss: nan, Val MAE: nan
Epoch 17/20 -> Train Loss: nan, Val Loss: nan, Va